# Pipeline de Análise e Projeção de Preços da Cesta Básica no Brasil (2020–2030)

**TCC — Engenharia de Software**

Este notebook integra todo o pipeline de ETL, análise exploratória, modelagem preditiva (SARIMA e Prophet) e projeções de cenários para o custo da cesta básica nas 27 capitais brasileiras, com **destaque para São Luís/MA**.

> **Objetivo:** Gerar visualizações de alta qualidade para apresentação em slides, demonstrando a evolução histórica e projeções até 2030.

## 1. 📦 Importações e Configuração

Nesta seção configuramos o ambiente, importamos os módulos do pipeline e definimos os caminhos de saída. Todos os gráficos serão salvos automaticamente em `outputs/graficos/` para uso nos slides da apresentação.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Adicionar src ao path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

from extract import load_data
from transform import transformar, calcular_media_nacional, obter_serie_agregada
from load import executar_pipeline, salvar_csv
from eda import executar_eda
from model import executar_modelagem, avaliar_modelos

plt.style.use("seaborn-v0_8-whitegrid")

OUTPUT_GRAFICOS = PROJECT_ROOT / "outputs" / "graficos"
OUTPUT_GRAFICOS.mkdir(parents=True, exist_ok=True)

print(f"Projeto: {PROJECT_ROOT}")
print(f"Gráficos serão salvos em: {OUTPUT_GRAFICOS}")

## 2. 🔄 ETL — Extração e Transformação dos Dados

O pipeline começa pela **extração** dos dados via `extract.py`. Por padrão, utilizamos dados **simulados** baseados em valores reais do DIEESE. Para trocar por dados reais, basta editar a função `load_data()` em `src/extract.py`.

Em seguida, aplicamos transformações: interpolação de ausentes, cálculo de variações, identificação de outliers e criação de features para modelagem.

In [ ]:
# Executar pipeline ETL completo
df = executar_pipeline()

print(f"\nShape dos dados: {df.shape}")
print(f"Período: {df['data'].min().strftime('%b/%Y')} a {df['data'].max().strftime('%b/%Y')}")
print(f"Capitais: {df['capital'].nunique()}")
df.head(10)

In [ ]:
# Visão geral dos dados transformados
print("Estatísticas do custo (R$):")
df.groupby('capital')['custo'].agg(['mean', 'min', 'max']).round(2).head(10)

In [ ]:
# Foco: São Luís em janeiro/2020 e março/2026
sl = df[df['capital'] == 'São Luís']
jan2020 = sl[sl['data'] == '2020-01-01']['custo'].values[0]
mar2026 = sl[sl['data'] == '2026-03-01']['custo'].values[0]
print(f"São Luís — Jan/2020: R$ {jan2020:,.2f}")
print(f"São Luís — Mar/2026: R$ {mar2026:,.2f}")
print(f"Variação acumulada: {((mar2026/jan2020)-1)*100:.1f}%")

## 3. 📊 EDA — Análise Exploratória de Dados

A análise exploratória revela padrões temporais, sazonalidade e diferenças regionais no custo da cesta básica. Todos os gráficos são exportados em PNG (150 dpi) para os slides.

### 3.1 Visão Nacional

O gráfico de evolução histórica mostra todas as 27 capitais, destacando **São Paulo** (vermelho), **São Luís** (verde) e a **média nacional** (azul tracejado). Observa-se a tendência de alta contínua, acelerada pelo choque da COVID-19 em 2020.

In [ ]:
# Gerar todos os gráficos de EDA
caminhos_eda = executar_eda(df)

from IPython.display import Image, display
for nome, caminho in caminhos_eda.items():
    print(f"\n--- {nome} ---")
    display(Image(filename=str(caminho)))

### 3.2 Foco: São Luís/MA

São Luís apresenta um dos menores custos entre as capitais, posicionando-se abaixo da média nacional. A decomposição da série revela tendência de crescimento, sazonalidade anual (picos no início do ano) e resíduos compatíveis com ruído de mercado.

In [ ]:
# Série de São Luís
serie_sl = obter_serie_agregada(df, capital='São Luís')
serie_sl.plot(x='data', y='custo', figsize=(12, 5), title='Custo da Cesta Básica — São Luís/MA', color='#1a5c38')
plt.ylabel('Custo (R$)')
plt.xlabel('Data')
plt.tight_layout()
plt.show()

### 3.3 Análise Regional

O boxplot regional evidencia que o **Sudeste** e o **Sul** concentram os maiores custos, enquanto o **Nordeste** e o **Norte** apresentam valores mais baixos. São Luís, no Nordeste, está entre as capitais mais acessíveis.

In [ ]:
# Custo médio por região
df.groupby('regiao')['custo'].mean().sort_values(ascending=False).round(2)

## 4. 🤖 Modelagem Preditiva

Utilizamos dois modelos de séries temporais para prever o custo da cesta básica:

- **SARIMA** (Seasonal ARIMA): modelo estatístico clássico com busca automática de parâmetros via AIC
- **Prophet** (Facebook): modelo aditivo com sazonalidade anual/mensal e feriados brasileiros

**Período de treino:** Jan/2020 a Dez/2024  
**Período de teste:** Jan/2025 a Mar/2026

### 4.1 SARIMA

O SARIMA captura tendência e sazonalidade através de componentes autoregressivos e de médias móveis, com período sazonal de 12 meses.

### 4.2 Prophet

O Prophet decompõe a série em tendência, sazonalidades múltiplas e efeitos de feriados, sendo robusto a dados faltantes e mudanças de tendência.

### 4.3 Comparação de Métricas

Avaliamos os modelos com **MAE** (erro absoluto médio), **RMSE** (raiz do erro quadrático médio) e **MAPE** (erro percentual absoluto médio).

In [ ]:
# Avaliar e exibir métricas
df_metricas = avaliar_modelos(df)
df_metricas

In [ ]:
# Tabela pivotada de métricas
df_metricas.pivot_table(index=['serie', 'modelo'], columns='metrica', values='valor').round(2)

## 5. 🔮 Projeções 2026–2030

Com base no modelo Prophet, projetamos o custo da cesta básica de **abril/2026 a dezembro/2030** em três cenários de inflação para alimentos:

| Cenário | Inflação Anual |
|---------|---------------|
| Otimista | 3,0% |
| Moderado | 4,5% |
| Conservador | 6,0% |

### 5.1 São Luís — Três Cenários

Este é o **gráfico principal** da apresentação. Mostra a trajetória histórica e as três projeções, com faixa de incerteza entre os cenários otimista e conservador.

In [ ]:
# Executar modelagem completa (métricas + projeções + gráficos)
resultados = executar_modelagem(df)

In [ ]:
# Exibir gráfico principal — São Luís
from IPython.display import Image, display
display(Image(filename=str(PROJECT_ROOT / 'outputs' / 'graficos' / 'projecao_sao_luis_2020_2030.png')))

### 5.2 Projeção Nacional

A projeção da média nacional complementa a análise de São Luís, contextualizando a tendência do país como um todo.

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'outputs' / 'graficos' / 'projecao_media_nacional_2020_2030.png')))

In [ ]:
# Valores projetados em dez/2030 por cenário — São Luís
proj_sl = resultados['projecoes'][resultados['projecoes']['capital'] == 'São Luís']
dez2030 = proj_sl[proj_sl['data'] == '2030-12-01'][['cenario', 'custo_projetado']]
print("Projeção São Luís — Dez/2030:")
dez2030

## 6. 📋 Conclusões

### Principais achados:

1. **Tendência de alta:** O custo da cesta básica apresentou crescimento consistente de 2020 a 2026, com aceleração no período da pandemia.
2. **São Luís entre as mais acessíveis:** A capital maranhense mantém custos abaixo da média nacional, beneficiando consumidores locais.
3. **Sazonalidade marcante:** Picos em janeiro-fevereiro e quedas em abril-maio são observados em todas as regiões.
4. **Projeções até 2030:** Mesmo no cenário otimista (3% a.a.), o custo em São Luís deve superar R$ 850; no conservador (6% a.a.), pode ultrapassar R$ 1.000.
5. **Modelos complementares:** SARIMA e Prophet apresentam desempenho comparável no período de teste, validando as projeções.

### Arquivos gerados:

- `outputs/graficos/` — 7 PNGs para slides
- `outputs/metricas_modelos.csv` — métricas de avaliação
- `outputs/projecoes_2026_2030.csv` — projeções mensais por cenário
- `data/processed/` — dados processados (CSV + SQLite)

In [ ]:
# Listar todos os arquivos gerados
print("=== Gráficos ===")
for f in sorted(OUTPUT_GRAFICOS.glob('*.png')):
    print(f"  {f.name}")

print("\n=== Outputs ===")
for f in sorted((PROJECT_ROOT / 'outputs').glob('*.csv')):
    print(f"  {f.name}")